# Administrative Document Classification - Hugging Face Inference

Load the **complete merged model** from Hugging Face and run inference without loading PEFT or the original base model separately. First export the model with [`inference_colab.ipynb`](inference_colab.ipynb).

The model classifies document format and administrative purpose. It is not trained to extract every metadata field listed in the dataset card.

## Before you start

1. Set a Colab GPU runtime (T4 or better recommended).
2. Set `HF_MODEL_REPO_ID` in the configuration cell to the repository where you uploaded the merged model.
3. If the repository is private, add a Hugging Face **read** token to Colab Secrets as `HF_TOKEN`. For a public repository, the token can be left unset unless Hugging Face requires authentication for access under the model's terms.
4. Follow the applicable Gemma terms. Do not send confidential documents through the optional public Gradio share link.


In [ ]:
%pip -q install transformers accelerate huggingface_hub gradio

## Configure and authenticate


In [ ]:
HF_MODEL_REPO_ID = "YOUR_USERNAME/gemma-admin-document-classifier"

if "YOUR_USERNAME" in HF_MODEL_REPO_ID:
    raise ValueError("Set HF_MODEL_REPO_ID to the Hugging Face model repository you exported.")

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN", default=None)
except (ImportError, KeyError):
    hf_token = None

if hf_token:
    print("Hugging Face token found; private repositories are accessible.")
else:
    print("No token found; this works for public repositories that permit unauthenticated access.")

## Load the full model

Unlike the adapter-only workflow, this model repository contains the merged weights, so it can be loaded directly with Transformers.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

if not torch.cuda.is_available():
    print("No GPU detected. CPU inference may be very slow; choose a GPU runtime for practical use.")

dtype = torch.float16 if torch.cuda.is_available() else torch.float32
tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_REPO_ID, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    HF_MODEL_REPO_ID,
    torch_dtype=dtype,
    device_map="auto",
    token=hf_token,
)
model.eval()
print("Full model loaded.")

## Classify a document


In [ ]:
def classify_document(document_text: str) -> str:
    """Generate document-format and administrative-purpose classifications."""
    if not document_text or not document_text.strip():
        raise ValueError("Enter document text before running inference.")

    prompt = f"""Analyze the document below and fill in the classification metadata.

Text: {document_text.strip()}

Document Format:"""
    inputs = tokenizer(prompt, return_tensors="pt")
    input_device = model.get_input_embeddings().weight.device
    inputs = {name: tensor.to(input_device) for name, tensor in inputs.items()}

    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = generated[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


sample_document = (
    "All faculty members are requested to attend the sensitization program "
    "on Sustainable Development Goals in the main auditorium this Thursday at 10:00 AM."
)
print(classify_document(sample_document))

## Optional: launch a Gradio interface

The generated public link works only while the Colab session remains active. Anyone with the link may be able to submit text.

In [ ]:
import gradio as gr

demo = gr.Interface(
    fn=classify_document,
    inputs=gr.Textbox(lines=12, label="Document text"),
    outputs=gr.Textbox(lines=6, label="Generated classification"),
    title="Administrative Document Classifier",
    description="Classifies document format and administrative purpose using the merged Gemma model.",
)
demo.launch(share=True)